# Exercises

:::{admonition} How to use these exercises
:class: note
Attempt each from scratch in the empty code cell; the distributed hand-out omits the solutions. Import geopandas as `gpd`, keep track of the crs at every step, and measure only after reprojecting to a metric crs.
:::

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [ ]:
# Your solution here

## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [ ]:
# Your solution here

## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [ ]:
# Your solution here

## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [ ]:
# Your solution here

## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [ ]:
# Your solution here

## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [ ]:
# Your solution here

## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
# Your solution here

## Exercise 8: Hurricane Florence — tracking a real storm

The [National Hurricane Center](https://www.nhc.noaa.gov/) issues an advisory for every active tropical system; this exercise uses the archived advisory track for Hurricane Florence (2018), together with a [US Census Bureau](https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html) cartographic boundary file for the 50 states.

Everything you need is from this subchapter: building a GeoDataFrame from coordinates, CRS/reprojection, layered plotting, and buffers with a spatial predicate.

In [ ]:
# Pre-supplied: download and cache the two real data files.
import pooch

track_path = pooch.retrieve(
    url="https://github.com/gse-unil/2026_MLEES_book/blob/main/data/part-I/florence.csv",
    known_hash="sha256:385691583ee41682a1c905e042c56ea362609fb04645934bdda7911c55c8b63f",
    fname="florence.csv",
    path=pooch.os_cache("mlees"),
)
states_path = pooch.retrieve(
    url="https://github.com/gse-unil/2026_MLEES_book/blob/main/data/part-I/gz_2010_us_040_00_5m.json",
    known_hash="sha256:7a8c022e063a34a83f35984cde6c81992ece5983f8cc4459ed02e40687739573",
    fname="us_states.geojson",
    path=pooch.os_cache("mlees"),
)

**Step 1.** Read `states_path` with `gpd.read_file` and `track_path` with `pd.read_csv` (parse `Date` as a date). Print each one's shape.

In [ ]:
# Your solution here
# Hint: gpd.read_file(states_path)
# Hint: pd.read_csv(track_path, parse_dates=["Date"])

**Step 2.** The track's `Long` column is stored as a positive "degrees west" magnitude, not a signed longitude — a common quirk of NHC advisory data; check a few values against a map before trusting a sign. Build a GeoDataFrame from the track with `gpd.points_from_xy(-track["Long"], track["Lat"])`, and give it the WGS84 crs (`EPSG:4326`). Print its crs and geometry type.

In [ ]:
# Your solution here
# Hint: gpd.GeoDataFrame(track, geometry=gpd.points_from_xy(-track["Long"], track["Lat"]), crs="EPSG:4326")

**Step 3.** Before filtering or plotting anything, check what kind of geometry each layer actually holds — a states boundary file is not guaranteed to be pure `Polygon` (a state with offshore islands can come back as `MultiPolygon`). Print the distinct geometry types in `states` and in the track GeoDataFrame.

In [ ]:
# Your solution here
# Hint: a GeoDataFrame's .geom_type gives one label per row; .unique() collapses it to
#       the distinct values present

**Step 4.** Restrict `states` to the contiguous 48 states by excluding `"Alaska"` and `"Hawaii"` from the `NAME` column.

In [ ]:
# Your solution here
# Hint: states[~states["NAME"].isin(["Alaska", "Hawaii"])]

**Step 5.** Plot the filtered states as a base map, with the hurricane track's points layered on top of the same axes. Label the axes and give the plot a title.

In [ ]:
# Your solution here
# Hint: lower48.plot(ax=ax, color=..., edgecolor=...) first, then track_points.plot(ax=ax, ...)
# Hint: both layers must share the same crs to line up correctly

**Step 6.** A storm's damaging wind field extends roughly 50 km from its center at this stage of its life. Reproject both the filtered states and the track to a metric crs (`EPSG:5070`, Contiguous Albers Equal Area), buffer the track by 50 km, and report which states intersect that buffer.

In [ ]:
# Your solution here
# Hint: .to_crs(5070) on both layers before measuring anything, exactly as earlier in this subchapter
# Hint: GeoSeries.buffer(50_000) on the reprojected track, then .union_all() to merge it into one shape
# Hint: a state "intersects" the buffer if states_m.intersects(buffer_geometry) is True